In [ ]:
!pip install faiss-cpu gdown h5py

In [ ]:
import json
import os
import pickle
import sys

import faiss
import gdown
import h5py
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tnrange, tqdm, tqdm_notebook

%matplotlib inline

# Seminar: kNN index and quantization

## 1. Load embeddings

In [ ]:
file_id = "1-1t1sbQXXyB-qZr-czqv2UJ4E5BQ7McI"
destination = "lastftm_factors.h5"
download_url = f"https://drive.google.com/uc?id={file_id}"

gdown.download(download_url, destination, quiet=False)

In [ ]:
h5f = h5py.File("lastftm_factors.h5", "r")
item_factors = h5f["items"][:]
user_factors = h5f["users"][:]
h5f.close()

In [ ]:
item_factors.shape

In [ ]:
user_factors.shape

## 2. Neighbours search with FAISS

[Faiss docs](https://faiss.ai/)

In [ ]:
%%time
ip_index = faiss.IndexFlatIP(
    item_factors.shape[1]
)  # Create index Exact Search for Inner Product

ip_index.add(item_factors)  # Add document embeddings to index
print(ip_index.ntotal)

In [ ]:
k = 10  # Choose 10 nearest neighbours
n_users = 100  # 100 users is real example of request batch

user_embeddings = user_factors[:n_users]

In [ ]:
%%timeit
# D - distances
# I - indices
dist_brute, index_brute = ip_index.search(user_embeddings, k)

In [ ]:
dist_brute.shape

In [ ]:
dist_brute, index_brute = ip_index.search(user_embeddings, k)

In [ ]:
dist_brute[:10]

In [ ]:
index_brute[:10]

In [ ]:
dist_brute.shape

Let's compare with vanilla method to find nearest neighbours:

In [ ]:
# too long because we sort full array
product = user_embeddings.dot(item_factors.T)
exact_indexes = np.argsort(product, axis=1)[:, : -k - 1 : -1]

In [ ]:
%%time
product = user_embeddings.dot(item_factors.T)
neighbours = np.partition(-1 * product, k, axis=1)[:, :k]
exact_distances = -1 * np.sort(neighbours, axis=1)

In [ ]:
np.allclose(dist_brute, exact_distances)

In [ ]:
np.equal(index_brute, exact_indexes).mean()

## 3. HNSW

<img src="images/hnsw.png" alt="drawing" width="500"/>

**HNSW** (Hierarchical navigable small world) - graph-based approximate nearest neighbor algorithm.

In [ ]:
%%time
neighbours = 50


hnsw_index = faiss.IndexHNSWFlat(
    item_factors.shape[1], neighbours, faiss.METRIC_INNER_PRODUCT
)  # Create HNSW index

hnsw_index.hnsw.efSearch = 1600
hnsw_index.hnsw.m = 16
hnsw_index.hnsw.efConstruction = 1000

faiss.normalize_L2(item_factors)
hnsw_index.add(item_factors)

In [ ]:
%%time
dist_hnsw, index_hnsw = hnsw_index.search(user_embeddings, k)

In [ ]:
%%time
# too long because we sort full array
product = user_embeddings.dot(item_factors.T)
exact_indexes = np.argsort(product, axis=1)[:, : -k - 1 : -1]

In [ ]:
index_hnsw[:1]

In [ ]:
dist_hnsw[:1]

In [ ]:
np.equal(index_hnsw, exact_indexes).mean()

Algorithm is not tuned for the dataset, so results are not the best. But you can find optimal parameters for 99% or even 99.9%.

## Extra: Nearest neighbours using KDTree

In [ ]:
from sklearn.neighbors import KDTree

In [ ]:
tree = KDTree(item_factors, leaf_size=5)
dist_kd, ind_kd = tree.query(user_embeddings, k=k)
print(ind_kd)

In [ ]:
np.equal(ind_kd, exact_indexes).mean()

## 4. Quantization

<img src="images/quant.png" alt="drawing" width="800"/>

In [ ]:
# first coordinates of item embeddings
item_factors[:, 0]

Let's look at distribution of first coordinates:

In [ ]:
plt.hist(item_factors[:, 0], bins=255)
plt.grid()
plt.show()

In [ ]:
plt.hist(user_factors[:, 0], bins=255)
plt.grid()
plt.show()

### a. Percentile quantizations

**Steps:**
1. **Compute Percentiles**:  
   Divide the embedding values into \(k\) bins (e.g., 256) by calculating percentile values for each dimension.

2. **Quantization**:  
   Map each value in the embeddings to its corresponding bin. Clip the bin indices to the range \([0, k-1]\).

3. **Restoration**:  
   Approximate the original values by replacing each quantized bin index with its corresponding percentile value.


Firstly, let's do it for **items**:

In [ ]:
item_factors.shape

In [ ]:
# sample.shape

In [ ]:
# sample_indexes

In [ ]:
sample_size = 100_000
item_percentiles = []
quanitles = np.linspace(0, 100, 256)

sample_indexes = np.random.randint(len(item_factors), size=sample_size)
sample = item_factors[sample_indexes]

thesholds = np.percentile(sample, quanitles, axis=0)
thesholds.shape

In [ ]:
thesholds

In [ ]:
item_percentiles = np.vstack(thesholds)

In [ ]:
def perc_quantize(v, percentiles):
    res = []
    for i in range(v.shape[1]):
        res.append(np.digitize(v[:, i], percentiles[:, i]))
    res = np.vstack(res).T
    res = np.clip(res, 0, 255)
    return res

In [ ]:
def perc_restore(q, percentiles):
    res = []
    for i in range(q.shape[1]):
        res.append(percentiles[q[:, i], i])
    return np.vstack(res).T

In [ ]:
# ideal plot: y = x
restored_vectors = perc_restore(
    perc_quantize(item_factors[:100, :], item_percentiles), item_percentiles
)
plt.scatter(item_factors[:100, :], restored_vectors, s=1)
plt.title("Item percentile quantization")
plt.xlim(-0.01, 0.02)
plt.ylim(-0.01, 0.02)

And for **users**:

In [ ]:
N = 100000
user_percentiles = []
for perc in tqdm(np.linspace(0, 100, 256)):
    user_percentiles.append(np.percentile(user_factors[:N, :], perc, axis=0))

In [ ]:
user_percentiles = np.vstack(user_percentiles)

In [ ]:
# ideal plot: y = x
plt.scatter(
    user_factors[:100, :],
    perc_restore(
        perc_quantize(user_factors[:100, :], user_percentiles), user_percentiles
    ),
    s=1,
)
plt.title("User percentile quantization")

plt.xlim(-4, 4)
plt.ylim(-4, 4)

Then we can check the overlap of quantized embeddings and original embeddings:

In [ ]:
overlaps = []
for user_idx in tqdm(range(1, N, int(N / 100))):
    scores1 = item_factors[:N].dot(user_factors[user_idx, :])
    scores2 = item_factors[:N].dot(
        perc_restore(
            perc_quantize(user_factors[user_idx : user_idx + 1, :], user_percentiles),
            user_percentiles,
        )[0]
    )
    order1 = np.argsort(scores1)[::-1][:100]
    order2 = np.argsort(scores2)[::-1][:100]
    overlaps.append(len(set(order1) & set(order2)))
print("MEAN:", np.mean(overlaps))

### b. MinMax quantization

**Steps:**
1. **Range:**
   
   First, determine the range of values by calculating the 1st and 99th percentiles of the embedding values. These percentiles represent the lower and upper bounds of the data, respectively, and they define the range over which the data will be quantized.

2. **Quantize:**
   
   The quantization process scales the embedding values to a [0, 255] range. This is done by first normalizing the data within the defined range, then scaling it to fit the target range (0 to 255). Any values outside the range are clipped to ensure they fall within the [0, 255] bounds.

3. **Restore:**
   
   After quantization, the original values can be restored by reversing the scaling process. The quantized values are multiplied by a scaling factor and then shifted back by the lower bound. This restores the values to a continuous range similar to the original data.

**Quantization:** $q(x) = \frac{x - min}{max - min} \cdot 256$

**Restore:** $q^{-1}(x) = \frac{x(max - min)}{256} + min$


Let's start for **items**:

In [ ]:
item_low = np.percentile(item_factors[:N], 1, axis=0)
item_high = np.percentile(item_factors[:N], 99, axis=0)

item_step = (item_high - item_low + 1e-6) / 255.0

In [ ]:
def mm_quantize(value, low, high):
    return (
        np.minimum(np.maximum((value - low) / (high - low + 1e-6), 0), 1) * 255
    ).astype(np.uint8)


def mm_restore(q, l, s):
    return q.astype(np.float32) * s + l

In [ ]:
plt.scatter(
    item_factors[0:100, :],
    mm_restore(
        mm_quantize(item_factors[0:100, :], item_low, item_high), item_low, item_step
    ),
    s=1,
)
plt.title("Item MinMax quantization")

plt.xlim(-0.01, 0.02)
plt.ylim(-0.01, 0.02)

In [ ]:
overlaps = []
boo = mm_restore(
    mm_quantize(item_factors[:N], item_low, item_high), item_low, item_step
)
for user_idx in range(1, N, int(N / 100)):
    scores1 = item_factors[:N].dot(user_factors[user_idx, :])
    scores2 = boo.dot(user_factors[user_idx, :])
    order1 = np.argsort(scores1)[::-1][:100]
    order2 = np.argsort(scores2)[::-1][:100]
    overlaps.append(len(set(order1) & set(order2)))
print("MEAN:", np.mean(overlaps))

And for **users**:

In [ ]:
user_low = np.percentile(user_factors[:N], 1, axis=0)
user_high = np.percentile(user_factors[:N], 99, axis=0)

user_step = (user_high - user_low + 1e-6) / 255.0

In [ ]:
plt.scatter(
    user_factors[0:100, :],
    mm_restore(
        mm_quantize(user_factors[0:100, :], user_low, user_high), user_low, user_step
    ),
    s=1,
)
plt.title("User MinMax quantization")

plt.xlim(-4, 4)
plt.ylim(-4, 4)

In [ ]:
overlaps = []
for user_idx in range(1, N, int(N / 100)):
    scores1 = item_factors[:N].dot(user_factors[user_idx, :])
    scores2 = item_factors[:N].dot(
        mm_restore(
            mm_quantize(user_factors[user_idx : user_idx + 1, :], user_low, user_high),
            user_low,
            user_step,
        )[0]
    )
    order1 = np.argsort(scores1)[::-1][:100]
    order2 = np.argsort(scores2)[::-1][:100]
    overlaps.append(len(set(order1) & set(order2)))
print("MEAN:", np.mean(overlaps))